## 4.1 CART 决策树(分类) - 核心逻辑&手算法
CART（Classification and Regression Trees）是一种常用的决策树算法，适用于分类和回归问题。在分类问题中，CART 决策树使用基尼指数（Gini Index）来衡量数据集的纯度，并通过计算 Gini Decrease 来选择最佳分裂点。<br>
注意：CART使用二叉树结构，即每个节点只能分成两个子节点，这与ID3和C4.5等算法不同，后者可以使用多叉树结构。CART 通过递归地选择最佳特征和分裂点来构建决策树，直到满足停止条件（如达到最大深度、最小样本数等）。<br>

#### 1. CART 决策树的不纯度和目标函数

CART 使用基尼指数（Gini Index）来衡量数据集的纯度。基尼指数越小，数据集越纯。对于一个数据集 D，基尼指数定义如下：
$$ Gini(D) = 1 - \sum_{k=1}^{K} p_k^2 $$
其中，$p_k$ 是数据集中属于第 k 类的样本比例，K 是类别的总数。<br>
CART 选择分裂的目标是 Gini Decrease, 即通过分裂后基尼指数的减少量来选择最佳分裂点。对于一个特征 A 的一个可能的分裂点 s，Gini Decrease 定义如下：
$$\text{Gini Decrease} = Gini(D) - \left( \frac{|D_{left}|}{|D|} Gini(D_{left}) + \frac{|D_{right}|}{|D|} Gini(D_{right}) \right)$$
其中，$D_{left}$ 和 $D_{right}$ 分别是根据特征 A 的分裂点 s 将数据集 D 分成的左右子集。CART 选择使 Gini Decrease 最大的特征和分裂点作为当前节点的分裂条件。

#### 2. CART对于不同类型特征的处理方式
CART 决策树对于不同类型的特征（数值型和类别型）有不同的处理方式：
1. **高取值类别型特征（离散变量）**：对于类别型特征，CART 会尝试将类别划分成两组来计算 Gini Decrease。例如，如果特征 B 的取值为 {Red, Green, Blue}，CART 会尝试将类别划分成 {Red} 和 {Green, Blue}，{Green} 和 {Red, Blue}，{Blue} 和 {Red, Green} 来计算 Gini Decrease，并选择 Gini Decrease 最大的划分方式进行分裂。<br>
2. **数值型特征(连续变量)**：对于数值型特征，CART 会尝试所有可能的分裂点（通常是特征值的中点）来计算 Gini Decrease，并选择 Gini Decrease 最大的分裂点进行分裂。例如，如果特征 A 的取值为 {1, 2, 3}，CART 会尝试分裂点 s = 1.5 和 s = 2.5 来计算 Gini Decrease。
通过这种方式，CART 决策树能够有效地处理不同类型的特征，并选择最佳的分裂点来构建决策树，从而实现对数据的分类或回归任务。

#### 3. CART 决策树的建树过程
CART 决策树的建树过程可以分为以下几个步骤：
1. **选择最佳分裂特征和分裂点**：对于当前节点的数据集，CART 会计算每个特征的所有可能分裂点的 Gini Decrease，并选择 Gini Decrease 最大的特征和分裂点作为当前节点的分裂条件（阈值/子集合）。
2. **分裂数据集**：根据选择的特征和分裂点，将数据集分成两个子集：左子集和右子集。
3. **递归建树**：对左子集和右子集分别重复步骤 1 和 2，直到满足停止条件（如达到最大深度、最小样本数等）。
4. **生成叶子节点**：当满足停止条件时，当前节点成为叶子节点，叶子节点的类别标签通常是该节点数据集中样本数量最多的类别。<br>
通过以上步骤，CART 决策树能够逐步构建出一棵分类树，从而实现对数据的分类任务。CART 决策树的建树过程是一个递归的过程，通过不断选择最佳分裂特征和分裂点来构建树结构，直到满足停止条件为止。

#### 4. CART 决策树的手算案例

##### 4.1 数据准备

In [1]:
import pandas as pd
# 创建数据集
data = {
    "Weather": ["Sunny", "Sunny", "Overcast", "Rainy", "Rainy", "Rainy", "Overcast", "Sunny", "Sunny", "Rainy"],
    "Temperature": ["Hot", "Hot", "Hot", "Mild", "Cool", "Cool", "Mild", "Mild", "Cool", "Mild"],
    "Humidity": ["High", "High", "High", "High", "Normal", "Normal", "Normal", "High", "Normal", "Normal"],
    "Windy": ["False", "True", "False", "False", "False", "True", "True", "False", "False", "False"],
    "Play": ["No", "No", "Yes", "Yes", "Yes", "No", "Yes", "No", "Yes", "Yes"]
}
df = pd.DataFrame(data)
df

,Weather,Temperature,Humidity,Windy,Play
0,Sunny,Hot,High,False,No
1,Sunny,Hot,High,True,No
2,Overcast,Hot,High,False,Yes
3,Rainy,Mild,High,False,Yes
4,Rainy,Cool,Normal,False,Yes
5,Rainy,Cool,Normal,True,No
6,Overcast,Mild,Normal,True,Yes
7,Sunny,Mild,High,False,No
8,Sunny,Cool,Normal,False,Yes
9,Rainy,Mild,Normal,False,Yes


##### 4.2 对于根节点的特征选择
在根节点上，我们需要选择一个最优特征来划分数据集。我们将计算每个特征的信息增益，并选择信息增益最大的特征作为根节点的划分特征。

In [8]:
# 计算根节点的Gini
# 当前根节点一共有10个样本，其中Yes为6个，No为4个
Gini_root = 1 - (6/10)**2 - (4/10)**2
Gini_root

0.48

###### 4.2.1 候选特征 - Humidity (只有2个取值，High和Normal，适合用CART的二叉树结构)

In [14]:
# 对于特征Humidity， 我们需要计算每个取值的子集的Gini，并计算加权平均Gini
# High子集，一共有5个样本，其中Yes有2个，No有3个
G_high = 1 - (2/5)**2 - (3/5)**2 # 值为0.48
# Normal子集，一共有5个样本，其中Yes有4个， No有1个
G_normal = 1 - (4/5)**2 - (1/5)**2 # 值为0.32
# 计算加权平均Gini
G_humidity = (5/10)*G_high + (5/10)*G_normal # 值为0.4
# 计算Gini Decrease
G_Decrease_humidity = Gini_root - G_humidity # 0.48-0.4=0.08
G_Decrease_humidity

0.08000000000000007

###### 4.2.2 候选特征 - Windy (只有2个取值，High和Normal，适合用CART的二叉树结构)

In [15]:
# 对于特征Windy，我们需要计算每个取值的子集的Gini，并计算加权平均Gini
# False子集，一共有7个样本，其中Yes有5个， No有2个
G_False = 1 - (5/7)**2 - (2/7)**2
# True子集，一共有3个样本，其中Yes有1个， No有2个
G_True = 1 - (1/3)**2 - (2/3)**2
# 计算加权平均Gini
G_windy = (7/10)*G_False + (3/10)*G_True
# 计算Gini Decrease
G_Decrease_windy = Gini_root - G_windy
G_Decrease_windy

0.06095238095238098

###### 4.2.3 候选特征 - Weather (有3个取值，Sunny、Overcast和Rainy，需要手动进行二分叉分裂)

In [19]:
# 由于Weather有三个子集，但是CART只支持二分叉分裂，我们需要手动尝试不同的二分叉方式来计算Gini Decrease，并选择Gini Decrease最大的方式作为分裂方式。

# 方式1: {sunny} + {overcast, rainy}
# Sunny子集： 一共有5个样本，其中Yes为2个，No为3个
G_sunny = 1 - (2/5)**2 - (3/5)**2
# Overcast + rainy子集： 一共5个样本，其中Yes为4个，No为1个
G_overcast_rain = 1 - (4/5)**2 - (1/5)**2
# 计算加权平均Gini
G_weather_1 = (5/10)*G_sunny + (5/10)*G_overcast_rain
# 计算Gini Decrease
Gini_Decrease_weather_1 = Gini_root - G_weather_1
print("方式1的Gini Decrease:", Gini_Decrease_weather_1)

# 方式2: {overcast} + {sunny, rainy}
# Overcast子集： 一共2个样本，其中Yes为2个，No为0个
G_overcast = 1 - (2/2)**2 - (0/2)**2
# Sunny + rainy子集： 一共8个样本，其中Yes为4个，No为4个
G_sunny_rain = 1 - (4/8)**2 - (4/8)**2
# 计算加权平均Gini
G_weather_2 = (2/10)*G_overcast + (8/10)*G_sunny_rain
# 计算Gini Decrease
Gini_Decrease_weather_2 = Gini_root - G_weather_2
print("方式2的Gini Decrease:", Gini_Decrease_weather_2)

# 方式3: {rainy} + {sunny, overcast}
# Rainy子集： 一共3个样本， 其中Yes为2个， No为1个
G_rainy = 1 - (2/3)**2 - (1/3)**2
# Sunny + overcast子集： 一共7个样本，其中Yes为4个，No为3个
G_sunny_overcast = 1 - (4/7)**2 - (3/7)**2
# 计算加权平均Gini
G_weather_3 = (3/10)*G_rainy + (7/10)*G_sunny_overcast
# 计算Gini Decrease
Gini_Decrease_weather_3 = Gini_root - G_weather_3
print("方式3的Gini Decrease:", Gini_Decrease_weather_3)

方式1的Gini Decrease: 0.08000000000000007
方式2的Gini Decrease: 0.07999999999999996
方式3的Gini Decrease: 0.003809523809523707


###### 4.2.4 候选特征 - Temperature (有3个取值，Hot、Mild和Cool，需要手动进行二分叉分裂)

In [20]:
# 方式1: {Hot} + {Mild, Cool}
# Hot子集： 一共有3个样本，其中Yes为1个，No为2个
G_hot = 1 - (1/3)**2 - (2/3)**2
# Mild + Cool子集： 一共7个样本，其中Yes为5个，No为2个
G_mild_cool = 1 - (5/7)**2 - (2/7)**2
# 计算加权平均Gini
G_temperature_1 = (3/10)*G_hot + (7/10)*G_mild_cool
# 计算Gini Decrease
Gini_Decrease_temperature_1 = Gini_root - G_temperature_1
print("方式1的Gini Decrease:", Gini_Decrease_temperature_1)

# 方式2: {Mild} + {Hot, Cool}
# Mild子集： 一共4个样本，其中Yes为3个， No为1个
G_mild = 1 - (3/4)**2 - (1/4)**2
# Hot + Cool子集： 一共6个样本，其中Yes为3个，No为3个
G_hot_cool = 1 - (3/6)**2 - (3/6)**2
# 计算加权平均Gini
G_temperature_2 = (4/10)*G_mild + (6/10)*G_hot_cool
# 计算Gini Decrease
Gini_Decrease_temperature_2 = Gini_root - G_temperature_2
print("方式2的Gini Decrease:", Gini_Decrease_temperature_2)

# 方式3: {Cool} + {Hot, Mild}
# Cool子集： 一共3个样本，其中Yes为2个， No为1个
G_cool = 1 - (2/3)**2 - (1/3)**2
# Hot + Mild子集： 一共7个样本，其中Yes为4个，No为3个
G_hot_mild = 1 - (4/7)**2 - (3/7)**2
# 计算加权平均Gini
G_temperature_3 = (3/10)*G_cool + (7/10)*G_hot_mild
# 计算Gini Decrease
Gini_Decrease_temperature_3 = Gini_root - G_temperature_3
print("方式3的Gini Decrease:", Gini_Decrease_temperature_3)

方式1的Gini Decrease: 0.06095238095238098
方式2的Gini Decrease: 0.02999999999999997
方式3的Gini Decrease: 0.003809523809523707


##### 4.3 选择最佳分裂特征和分裂点
 - 经过计算得知，特征Humidity的Gini Decrease为0.08，
 - 特征Windy的Gini Decrease为0.02，
 - 特征Weather的三种分裂方式的Gini Decrease分别为0.08、0.04和0.06，
 - 特征Temperature的三种分裂方式的Gini Decrease分别为0.02、0.04和0.06。
 - 因此，我们选择特征Humidity作为根节点的分裂特征，因为它具有最大的Gini Decrease（0.08）。

#### 5. 分裂得到的结果：
根节点：Humidity
 - Humidity=High的子节点：Weather
 - Humidity=Normal的子节点：Windy